# Setup Environment (Google Colab)
Chạy các block dưới đây để thiết lập môi trường Colab. Thiết lập này bao gồm:
- Bật code JS chống tự động ngắt kết nối (Anti-Disconnect).
- Liên kết Google Drive và Symlink thư mục logs để file checkpoint được lưu thẳng vào Drive.
- Khả năng tự động tìm lại Checkpoint cũ nếu quá trình train bị gián đoạn.

In [ ]:
# 1. Chống ngắt kết nối Colab (Chạy nền JS)
import IPython
display(IPython.display.Javascript('''
 function ClickConnect(){
   console.log("Clicking Connect để chống ngắt kết nối..."); 
   document.querySelector("colab-connect-button").click() 
 }
 setInterval(ClickConnect, 60000)
'''))
print("Đã bật chống ngắt kết nối tự động. Colab sẽ tự click sau mỗi 60 giây.")

In [ ]:
# 2. Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục dự án trên Drive nếu chưa có để chứa logs và checkpoints
import os
os.makedirs('/content/drive/MyDrive/palm_experiments/logs/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/palm_experiments/logs/tensorboard', exist_ok=True)

In [ ]:
# 3. Clone mã nguồn từ GitHub
# Đổi link GitHub của bạn tại đây:
GIT_URL = "https://github.com/your-username/palm.git"

if not os.path.exists('/content/palm'):
    !git clone {GIT_URL} /content/palm
else:
    print("Thư mục đã tồn tại, tiến hành cập nhật code mới nhất...")
    %cd /content/palm
    !git pull

%cd /content/palm

In [ ]:
# 4. Symlink thư mục logs vào Google Drive để tự động sao lưu an toàn!
# Nếu Colab bị ngắt, file checkpoints và log file vẫn an toàn trên Drive của bạn.
!rm -rf /content/palm/logs
!ln -s /content/drive/MyDrive/palm_experiments/logs /content/palm/logs
print("Đã liên kết (symlink) thư mục logs sang Google Drive thành công!")

---
# Train Generative Model (VAE + Contrastive)
Hệ thống sẽ tự động phát hiện nếu có checkpoint bị ngắt quãng trước đó trên Drive và tự động Resume.

In [ ]:
import sys
import os
import glob

# Đưa thư mục hiện tại vào sys.path
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.append(project_root)

# Tìm file checkpoint gần nhất trên Drive để tiếp tục quá trình train
checkpoints = glob.glob('/content/drive/MyDrive/palm_experiments/logs/checkpoints/*_last.pth')
resume_cmd = ""
if checkpoints:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"[INFO] Tìm thấy checkpoint bị ngắt quãng, hệ thống sẽ tự động resume từ: {latest_checkpoint}")
    resume_cmd = f"--resume '{latest_checkpoint}'"
else:
    print("[INFO] Không tìm thấy checkpoint nào trên Drive. Bắt đầu train mới từ đầu.")

# Tạo câu lệnh hoàn chỉnh
train_command = f"python tools/train_generative_model.py --config config/mnist.yaml {resume_cmd}"
print(f"\nĐang chạy: {train_command}")

# Khởi chạy file train
!{train_command}